<a href="https://colab.research.google.com/github/namii07/Namisha-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Day_04_Machine_Learning_and_Big_Data_PySpark_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MACHINE LEARNING

In [ ]:
import pandas as pd

In [ ]:
a=pd.read_csv("/content/SOCR-HeightWeight.csv")

In [ ]:
a.head()

,Index,Height(Inches),Weight(Pounds)
0,1,65.78331,112.9925
1,2,71.51521,136.4873
2,3,69.39874,153.0269
3,4,68.21660,142.3354
4,5,67.78781,144.2971


In [ ]:
x=a[["Height(Inches)"]]
y=a[["Weight(Pounds)"]]


In [ ]:
a.isnull()

,Index,Height(Inches),Weight(Pounds)
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
24995,False,False,False
24996,False,False,False
24997,False,False,False
24998,False,False,False


In [ ]:
a.isnull().any()

,0
Index,False
Height(Inches),False
Weight(Pounds),False


In [ ]:
a.isnull().sum()

,0
Index,0
Height(Inches),0
Weight(Pounds),0


In [ ]:
from sklearn.model_selection import train_test_split

#Splits the data into training and testing set - 4 components.
#test_size=0.2 -> means the percentage of data to be used for testing from the test set.
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
x_train

,Height(Inches)
11734,65.26077
1862,71.68649
12163,67.99030
19086,67.60650
16075,66.91739
...,...
1678,65.19039
16390,72.18220
8303,69.01541
8887,69.06700


In [ ]:
from sklearn.linear_model import LinearRegression
model =LinearRegression()
model.fit(x_train,y_train) #use(fit) the model to train the data

LinearRegression()

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error , r2_score

In [ ]:
mean_absolute_error(y_test,y_pred)

7.993325815240241

In [ ]:
mean_squared_error(y_test,y_pred)

100.07639378199183

In [ ]:
r2_score(y_test,y_pred)

0.2630493142744561

In [ ]:
import joblib

In [ ]:
joblib.dump(model,"linear.pkl")

['linear.pkl']

BIG DATA AND PySPARK ARCHITECTURE

In [ ]:
!pip install pyspark --quiet

print('PySpark Installation Completed!')


PySpark Installation Completed!


In [ ]:
from pyspark.sql import SparkSession

from pyspark.sql import functions as F

from pyspark.sql.functions import year, month, to_date , col , round as spark_round

import matplotlib.pyplot as plt

import pandas as pd

import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName('Day4_BigData_Sales') \
    .config('spark.sql.adaptive.enabled','true') \
    .getOrCreate()

print(f'Spark version : {spark.version}')
print(f'Spark Session : ACTIVE')
print(f'Application   : {spark.sparkContext.appName}')


Spark version : 4.0.2
Spark Session : ACTIVE
Application   : Day4_BigData_Sales


In [ ]:
df_bronze=spark.read \
  .option('header', 'true') \
  .option('interSchema', 'true') \
  .csv('large_sales_data.csv')

print('=== BRONZE LAYER - Raw Data ===')
print(f"Rows : {df_bronze.count()}")
print(f"Columns : {len(df_bronze.columns)}")
print(f"Names : {df_bronze.columns}")
print()
df_bronze.printSchema()

=== BRONZE LAYER - Raw Data ===
Rows : 5000
Columns : 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
print('First 5 rows:')
df_bronze.show(5, truncate=False)

print('\n Basic statistics for numeric columns:')
df_bronze.select('quantity' , 'unit_price' , 'revenue').describe().show()


First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [24]:
df_bronze.write \
   .mode('overwrite') \
   .parquet('sales_bronze.parquet')
print('Bronze parquet saved: sales_bronze_parquet')



Bronze parquet saved: sales_bronze_parquet
